In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import r2_score, mean_absolute_percentage_error, mean_squared_error
import joblib

# 1. ЗАГРУЗКА ДАННЫХ
df_train_raw = pd.read_csv("drum_dataset_ml.csv")
df_test_raw = pd.read_csv("drum_dataset_inference.csv")

def engineer_features_drum_v2(data):
    df = data.copy()
    
    # А. Унификация давления
    if 'Design gauge pressure KPAG' in df.columns:
        df['gauge_pres'] = df['Design gauge pressure KPAG']
    
    # Б. Исправление пропусков
    # Если давление не указано, для емкостей это чаще всего 0 (атмосферное)
    df['gauge_pres'] = df['gauge_pres'].fillna(0)
    
    # Если объем не указан, считаем его примерно по геометрии
    if 'liq_volume' in df.columns:
        vol_calc = np.pi * (df['ves_diameter']/2)**2 * df['ss_distance']
        df['liq_volume'] = df['liq_volume'].fillna(vol_calc)
    
    # В. Логарифмирование
    cols_to_log = ['liq_volume', 'ves_diameter', 'ss_distance', 'gauge_pres', 'volume_proxy', 'surface_area']
    for col in cols_to_log:
        if col in df.columns:
            df[f'log_{col}'] = np.log1p(df[col].clip(lower=0))
            
    return df

# 2. ПОДГОТОВКА
train_df = engineer_features_drum_v2(df_train_raw)
test_df = engineer_features_drum_v2(df_test_raw)

features = [
    'log_liq_volume', 'log_ves_diameter', 'log_ss_distance', 
    'log_gauge_pres', 'log_volume_proxy', 'log_surface_area', 
    'aspect_ratio'
]

X_train = train_df[features]
y_train = train_df["weight_kg"] # Передаем чистый вес, трансформация будет внутри
X_test = test_df[features]
y_test = test_df["weight_kg"]

# 3. ОБУЧЕНИЕ МОДЕЛИ
# Используем автоматическую логарифмическую трансформацию таргета
model = TransformedTargetRegressor(
    regressor=HistGradientBoostingRegressor(
        max_iter=1000,
        learning_rate=0.05,
        max_depth=6,
        l2_regularization=1.5,
        random_state=42
    ),
    func=np.log1p,
    inverse_func=np.expm1
)

model.fit(X_train, y_train)

# 4. ОЦЕНКА
def evaluate_metrics(model, X, y_true, label=""):
    y_pred = model.predict(X)
    
    print(f"\n=== {label} ===")
    print(f"R²:    {r2_score(y_true, y_pred):.4f}")
    print(f"MAPE:  {mean_absolute_percentage_error(y_true, y_pred)*100:.2f}%")
    print(f"RMSE:  {np.sqrt(mean_squared_error(y_true, y_pred)):.1f} кг")
    
    # Вывод нескольких примеров для проверки
    comparison = pd.DataFrame({'True': y_true, 'Pred': y_pred}).head(5)
    print("\nПримеры предсказаний:")
    print(comparison)

evaluate_metrics(model, X_test, y_test, "ТЕСТ (DRUM INFERENCE)")

# 5. СОХРАНЕНИЕ
joblib.dump(model, "drum_model_final.joblib")